# Email Spam Classification Platform — End-to-End Pipeline

Covers Phase 1 (EDA & Cleaning), Phase 2 (`transform_text()`), Phase 3 (TF-IDF + 7-Classifier Benchmarking), Phase 4 (Pickle Serialization & Fresh Verification), and PyTorch Bidirectional LSTM Training.

In [ ]:
import json
import pickle
import time
from pathlib import Path

import pandas as pd
from src.eda import clean_and_prepare_data, load_raw_dataset, plot_eda_charts
from src.preprocess import classify_threat_type, ensure_nltk_data, transform_text
from src.train import run_training_pipeline
from src.train_lstm import train_lstm_pipeline

## Phase 1 & 2: EDA, Deduplication, Label Encoding (`ham=0`, `spam=1`), and `transform_text()`

In [ ]:
ensure_nltk_data()
raw_df = load_raw_dataset()
clean_df, eda_summary = clean_and_prepare_data(raw_df)
plot_eda_charts(clean_df, output_dir=Path("artifacts"))
print(json.dumps(eda_summary, indent=2))
clean_df[["label", "target", "num_characters", "num_words", "transformed_text"]].head()

## Phase 3 & 4: TF-IDF (`max_features=3000`), 7-Classifier Benchmark, and Serialization

In [ ]:
report = run_training_pipeline()
pd.DataFrame(report["benchmarks"])[["model", "accuracy", "precision", "recall", "f1", "confusion_matrix", "selected"]]

## Fresh-Session Pickle Verification & Threat Type Identification

In [ ]:
with open("vectorizer.pkl", "rb") as vf:
    vec = pickle.load(vf)
with open("model.pkl", "rb") as mf:
    mnb = pickle.load(mf)

sample = "URGENT: Your bank account has been suspended. Verify your PIN at www.verify-login-secure.com"
t0 = time.perf_counter()
cleaned = transform_text(sample)
pred = int(mnb.predict(vec.transform([cleaned]))[0])
latency_ms = (time.perf_counter() - t0) * 1000.0
print("Prediction:", "Spam" if pred == 1 else "Not Spam", f"({latency_ms:.3f} ms)")
print("Threat Detail:", classify_threat_type(sample, pred))

## Deep Learning Extension: PyTorch Bidirectional LSTM (128-dim Embedding)

In [ ]:
lstm_summary = train_lstm_pipeline(epochs=12)
print(json.dumps(lstm_summary["best_metrics"], indent=2))